# 🔬 Raw Dataset Forensic Audit (STREAMING MODE)

**Problem:** 33M+ rows crash Colab RAM

**Solution:** Polars Lazy API + Streaming Engine

**Files:**
- `metabonet_public_train.parquet` (~33M rows)
- `metabonet_public_test.parquet`

## 1. Setup

In [ ]:
!pip install polars -q

from google.colab import drive
drive.mount('/content/drive')

In [ ]:
import polars as pl
import os

print(f"Polars: {pl.__version__}")
print("✅ Using STREAMING mode for low-memory processing")

In [ ]:
BASE_PATH = "/content/drive/MyDrive/DiabetesKDS/data/raw"

FILES = {
    "train": f"{BASE_PATH}/metabonet_public_train.parquet",
    "test": f"{BASE_PATH}/metabonet_public_test.parquet"
}

for name, path in FILES.items():
    exists = "✅" if os.path.exists(path) else "❌"
    print(f"{exists} {name}: {path}")

## 2. Lazy Scanning (NO RAM LOADING)

In [ ]:
# Columns we need (minimize memory footprint)
COLS = ['CGM', 'bolus', 'carbs', 'basal', 'age', 'weight']

def create_lazy_scan(path, label):
    """Create lazy scan with minimal columns and Float32 casting."""
    print(f"📂 Scanning {label}...")
    
    # Scan without loading
    lf = pl.scan_parquet(path)
    
    # Check available columns
    available = [c for c in COLS if c in lf.columns]
    print(f"   Available: {available}")
    
    # Select and cast to Float32 for memory savings
    lf = lf.select(available).cast({c: pl.Float32 for c in available})
    
    return lf, available

lf_train, train_cols = create_lazy_scan(FILES["train"], "TRAIN")
lf_test, test_cols = create_lazy_scan(FILES["test"], "TEST")

print("\n✅ Lazy scans created (no data loaded yet)")

In [ ]:
# Get row counts (streaming)
print("📊 Counting rows (streaming)...")

train_count = lf_train.select(pl.count()).collect(streaming=True).item()
test_count = lf_test.select(pl.count()).collect(streaming=True).item()

print(f"\n   Train: {train_count:,} rows")
print(f"   Test:  {test_count:,} rows")

## 3. Streaming Distribution Analysis

In [ ]:
def add_glucose_bin_lazy(lf):
    """Add glucose_bin column to lazy frame."""
    return lf.with_columns(
        pl.when(pl.col("CGM") < 70)
        .then(pl.lit("1_Hypo"))
        .when(pl.col("CGM") < 100)
        .then(pl.lit("2_Low_Normal"))
        .when(pl.col("CGM") < 150)
        .then(pl.lit("3_Normal"))
        .when(pl.col("CGM") < 250)
        .then(pl.lit("4_High"))
        .otherwise(pl.lit("5_Very_High"))
        .alias("glucose_bin")
    )

def analyze_streaming(lf, label, total_count):
    """Analyze distribution using streaming - ONE PASS."""
    print(f"\n📊 Analyzing {label} (streaming)...")
    
    # Add glucose bin and aggregate in one pass
    result = (
        add_glucose_bin_lazy(lf)
        .group_by("glucose_bin")
        .agg([
            pl.count().alias("count"),
            pl.col("bolus").mean().alias("mean_bolus"),
            (pl.col("bolus") > 0).sum().alias("bolus_positive_count"),
            pl.col("bolus").max().alias("max_bolus")
        ])
        .sort("glucose_bin")
        .collect(streaming=True)
    )
    
    # Add percentage
    result = result.with_columns(
        (pl.col("count") / total_count * 100).round(2).alias("pct")
    )
    
    print(result)
    return result

In [ ]:
# Run streaming analysis
train_dist = analyze_streaming(lf_train, "TRAIN", train_count)
test_dist = analyze_streaming(lf_test, "TEST", test_count)

In [ ]:
# Side-by-side comparison
print("\n" + "=" * 70)
print("📊 GLUCOSE BIN COMPARISON")
print("=" * 70)

bins = ['1_Hypo', '2_Low_Normal', '3_Normal', '4_High', '5_Very_High']

print(f"\n{'Bin':15} | {'Train %':>10} | {'Test %':>10} | {'Train Bolus':>12} | {'Test Bolus':>12}")
print("-" * 70)

for b in bins:
    t_row = train_dist.filter(pl.col("glucose_bin") == b)
    te_row = test_dist.filter(pl.col("glucose_bin") == b)
    
    t_pct = t_row["pct"].to_list()[0] if len(t_row) > 0 else 0
    te_pct = te_row["pct"].to_list()[0] if len(te_row) > 0 else 0
    t_bolus = t_row["mean_bolus"].to_list()[0] if len(t_row) > 0 else 0
    te_bolus = te_row["mean_bolus"].to_list()[0] if len(te_row) > 0 else 0
    
    diff = abs(t_pct - te_pct)
    flag = "⚠️" if diff > 5 else ""
    
    print(f"{b:15} | {t_pct:>10.2f} | {te_pct:>10.2f} | {t_bolus:>12.4f} | {te_bolus:>12.4f} {flag}")

## 4. Toxic Row Audit (Streaming)

In [ ]:
def count_toxic_streaming(lf, label, total_count):
    """Count toxic rows using streaming."""
    print(f"\n🧪 TOXIC ROW AUDIT: {label}")
    print("=" * 50)
    
    results = {}
    
    # 1. Hypo Hazard: CGM < 70 & Carbs == 0 & Bolus > 0
    hypo = (
        lf.filter(
            (pl.col("CGM") < 70) & 
            (pl.col("carbs") == 0) & 
            (pl.col("bolus") > 0)
        )
        .select(pl.count())
        .collect(streaming=True)
        .item()
    )
    results["Hypo Hazard (CGM<70)"] = hypo
    
    # 2. Low Normal + Bolus: 70 <= CGM < 100 & Carbs == 0 & Bolus > 0
    low_normal = (
        lf.filter(
            (pl.col("CGM") >= 70) &
            (pl.col("CGM") < 100) & 
            (pl.col("carbs") == 0) & 
            (pl.col("bolus") > 0)
        )
        .select(pl.count())
        .collect(streaming=True)
        .item()
    )
    results["Stable Low (70-100)"] = low_normal
    
    # 3. Low CGM + Bolus (broader): CGM < 110 & Carbs == 0 & Bolus > 0
    low_cgm = (
        lf.filter(
            (pl.col("CGM") < 110) & 
            (pl.col("carbs") == 0) & 
            (pl.col("bolus") > 0)
        )
        .select(pl.count())
        .collect(streaming=True)
        .item()
    )
    results["Low CGM (CGM<110)"] = low_cgm
    
    # Print results
    total_toxic = 0
    for scenario, count in results.items():
        pct = count / total_count * 100
        print(f"   {scenario}: {count:,} ({pct:.3f}%)")
        total_toxic = max(total_toxic, count)  # Use max since they overlap
    
    print(f"\n   📊 Worst Case Toxic: {low_cgm:,} rows")
    return results

In [ ]:
train_toxic = count_toxic_streaming(lf_train, "TRAIN", train_count)
test_toxic = count_toxic_streaming(lf_test, "TEST", test_count)

## 5. Zero Bolus Analysis

In [ ]:
# Count zero bolus (streaming)
print("\n📊 ZERO BOLUS ANALYSIS")
print("=" * 50)

train_zero = (
    lf_train
    .filter(pl.col("bolus") == 0)
    .select(pl.count())
    .collect(streaming=True)
    .item()
)

test_zero = (
    lf_test
    .filter(pl.col("bolus") == 0)
    .select(pl.count())
    .collect(streaming=True)
    .item()
)

train_zero_pct = train_zero / train_count * 100
test_zero_pct = test_zero / test_count * 100

print(f"   Train Zero Bolus: {train_zero:,} ({train_zero_pct:.1f}%)")
print(f"   Test Zero Bolus:  {test_zero:,} ({test_zero_pct:.1f}%)")
print(f"\n   Difference: {abs(train_zero_pct - test_zero_pct):.1f}%")

if abs(train_zero_pct - test_zero_pct) > 10:
    print("   ⚠️ SIGNIFICANT MISMATCH!")

In [ ]:
# Mean bolus (streaming)
print("\n📊 MEAN BOLUS COMPARISON")
print("=" * 50)

train_mean = (
    lf_train
    .select(pl.col("bolus").mean())
    .collect(streaming=True)
    .item()
)

test_mean = (
    lf_test
    .select(pl.col("bolus").mean())
    .collect(streaming=True)
    .item()
)

print(f"   Train Mean Bolus: {train_mean:.4f}")
print(f"   Test Mean Bolus:  {test_mean:.4f}")
print(f"   Ratio: {test_mean/train_mean:.2f}x" if train_mean > 0 else "")

## 6. Feature Statistics (Streaming)

In [ ]:
def get_feature_stats_streaming(lf, feature):
    """Get mean, std, max for a feature using streaming."""
    result = (
        lf
        .select([
            pl.col(feature).mean().alias("mean"),
            pl.col(feature).std().alias("std"),
            pl.col(feature).max().alias("max")
        ])
        .collect(streaming=True)
    )
    return result.row(0)

print("\n📉 FEATURE STATISTICS")
print("=" * 70)

features = ['CGM', 'bolus', 'carbs']
available_features = [f for f in features if f in train_cols]

print(f"\n{'Feature':10} | {'Train Mean':>12} | {'Test Mean':>12} | {'Train Std':>12} | {'Test Std':>12}")
print("-" * 70)

for f in available_features:
    t_stats = get_feature_stats_streaming(lf_train, f)
    te_stats = get_feature_stats_streaming(lf_test, f)
    
    diff = abs(t_stats[0] - te_stats[0]) / t_stats[0] * 100 if t_stats[0] != 0 else 0
    flag = "⚠️" if diff > 20 else ""
    
    print(f"{f:10} | {t_stats[0]:>12.2f} | {te_stats[0]:>12.2f} | {t_stats[1]:>12.2f} | {te_stats[1]:>12.2f} {flag}")

## 7. Summary Report

In [ ]:
print("\n" + "=" * 70)
print("📋 FORENSIC AUDIT SUMMARY (STREAMING MODE)")
print("=" * 70)

print(f"""
DATASET SIZES:
   Train: {train_count:,} rows
   Test:  {test_count:,} rows

ZERO BOLUS DISTRIBUTION:
   Train: {train_zero_pct:.1f}%
   Test:  {test_zero_pct:.1f}%
   Gap:   {abs(train_zero_pct - test_zero_pct):.1f}%

MEAN BOLUS:
   Train: {train_mean:.4f}
   Test:  {test_mean:.4f}

TOXIC ROWS (CGM<110, Carbs=0, Bolus>0):
   Train: {train_toxic['Low CGM (CGM<110)']:,}
   Test:  {test_toxic['Low CGM (CGM<110)']:,}
""")

print("\n🔍 KEY FINDINGS:")
if abs(train_zero_pct - test_zero_pct) > 10:
    print("   ⚠️ Train/Test have DIFFERENT zero-bolus ratios")
if test_mean < train_mean * 0.5:
    print("   ⚠️ Test mean bolus is <50% of Train")
if train_toxic['Hypo Hazard (CGM<70)'] > 0:
    print(f"   ⚠️ {train_toxic['Hypo Hazard (CGM<70)']:,} TOXIC rows in Train (Hypo+Bolus)")

print("\n" + "=" * 70)
print("✅ Audit completed without exceeding RAM limit")